# Preprocessing — Fashion Product Images

Notebook ini menghasilkan input siap-latih untuk **Model 1 (CNN tagger)**,
mengikuti PRD §4.4.1:

1. Parse `styles.csv` (dengan `on_bad_lines='skip'`).
2. Cleaning & pemetaan label ke `ml/configs/labels.yaml`.
3. Split 80/10/10 **stratified** (seed 42) → `splits/{train,val,test}.csv`.
4. Cache semua gambar sekali melalui `preprocess_pil` → `fashion_128.npy` (uint8 128×128×3).
5. Indeks utama `fashion_index.csv` yang menyelaraskan `npy_index` ↔ label ↔ split.

**Output** (dipakai notebook modeling):
```
ml/data/processed/fashion_128.npy
ml/data/processed/fashion_index.csv
ml/data/processed/splits/{train,val,test}.csv
ml/artifacts/fashion_prepare_report.json
```

## 0. Setup & parameter

In [1]:
import os, sys, json
from pathlib import Path
from collections import Counter

# Pastikan working directory = folder ml/ (tempat config_utils.py berada).
HERE = Path.cwd()
if not (HERE / "config_utils.py").exists():
    if (HERE / "ml" / "config_utils.py").exists():
        os.chdir(HERE / "ml")
    else:
        raise RuntimeError("Buka/run notebook ini dari folder ml/.")
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_rows", 80)
pd.set_option("display.width", 160)
plt.rcParams["figure.figsize"] = (12, 4)

import config_utils as cu

ARTIFACT_DIR = cu.ensure_dir(cu.artifact_dir())
PROCESSED_DIR = cu.ensure_dir(cu.fashion_processed_dir())
SPLITS_DIR = cu.ensure_dir(PROCESSED_DIR / "splits")
print("data     :", cu.to_posix(cu.data_dir()))
print("processed:", cu.to_posix(PROCESSED_DIR))

data     : C:/Kuliah/Semester 5/neurfit/ml/data
processed: C:/Kuliah/Semester 5/neurfit/ml/data/processed


In [2]:
# ---- Parameter (ubah bila perlu) ----
IMG_SIZE = 128
SEED = 42
OTHER_MAX = 4000
EXCLUDE_GENDER = ("Boys", "Girls")
MAX_SAMPLES = None        # isi angka (mis. 2000) untuk uji cepat
FORCE_REBUILD = False     # True = paksa buat ulang fashion_128.npy
print("IMG_SIZE", IMG_SIZE, "| SEED", SEED, "| OTHER_MAX", OTHER_MAX,
      "| MAX_SAMPLES", MAX_SAMPLES, "| FORCE_REBUILD", FORCE_REBUILD)

IMG_SIZE 128 | SEED 42 | OTHER_MAX 4000 | MAX_SAMPLES None | FORCE_REBUILD False


## 1. Label taxonomy (urutan = indeks softmax)

In [3]:
labels = cu.load_labels()
article_keys = [e["key"] for e in labels["article_type"]]
colour_keys = [e["key"] for e in labels["base_colour"]]
usage_keys = [e["key"] for e in labels["usage"]]

article_ds_to_key = {e["dataset"]: e["key"] for e in labels["article_type"] if e.get("dataset")}
article_to_slot = {e["key"]: e.get("slot") for e in labels["article_type"]}
colour_ds_to_key = {ds: e["key"] for e in labels["base_colour"] for ds in e["dataset"]}
usage_ds_to_key = {ds: e["key"] for e in labels["usage"] for ds in e["dataset"]}

print("labels version:", labels.get("version"))
print("article_type  (%d):" % len(article_keys), article_keys)
print("base_colour   (%d):" % len(colour_keys), colour_keys)
print("usage         (%d):" % len(usage_keys), usage_keys)
print("\narticle_to_slot:", article_to_slot)

labels version: labels-v1
article_type  (24): ['tshirt', 'shirt', 'top', 'sweatshirt', 'sweater', 'jacket', 'jeans', 'trousers', 'shorts', 'skirt', 'track_pants', 'casual_shoes', 'formal_shoes', 'sports_shoes', 'sandals', 'flip_flops', 'heels', 'flats', 'watch', 'belt', 'handbag', 'backpack', 'sunglasses', 'other']
base_colour   (14): ['black', 'white', 'grey', 'navy_blue', 'blue', 'brown', 'beige', 'red', 'green', 'pink', 'yellow', 'orange', 'purple', 'multi']
usage         (5): ['casual', 'smart_casual', 'formal', 'sports', 'party']

article_to_slot: {'tshirt': 'top', 'shirt': 'top', 'top': 'top', 'sweatshirt': 'top', 'sweater': 'top', 'jacket': 'top', 'jeans': 'bottom', 'trousers': 'bottom', 'shorts': 'bottom', 'skirt': 'bottom', 'track_pants': 'bottom', 'casual_shoes': 'shoes', 'formal_shoes': 'shoes', 'sports_shoes': 'shoes', 'sandals': 'shoes', 'flip_flops': 'shoes', 'heels': 'shoes', 'flats': 'shoes', 'watch': 'accessory', 'belt': 'accessory', 'handbag': 'accessory', 'backpack':

## 2. Parse & cleaning

In [4]:
raw_dir = cu.fashion_raw_dir()
styles_csv = raw_dir / "styles.csv"
images_dir = raw_dir / "images"
styles = pd.read_csv(styles_csv, on_bad_lines="skip")
print("raw rows:", len(styles))

STYLE_COLUMNS = ["id", "gender", "articleType", "baseColour", "usage"]


def clean_fashion(df, images_dir, *, exclude_gender=EXCLUDE_GENDER, other_max=OTHER_MAX,
                  seed=SEED, max_samples=None):
    stats = {}
    out = df[STYLE_COLUMNS].copy()

    out["id"] = pd.to_numeric(out["id"], errors="coerce")
    before = len(out)
    out = out[out["id"].notna()].copy()
    out["id"] = out["id"].astype("int64")
    stats["drop_bad_id"] = before - len(out)

    before = len(out)
    out = out[out["articleType"].notna() & out["baseColour"].notna()].copy()
    stats["drop_missing_core"] = before - len(out)

    before = len(out)
    out = out[~out["gender"].isin(list(exclude_gender))].copy()
    stats["drop_gender"] = before - len(out)

    ps = [images_dir / f"{i}.jpg" for i in out["id"]]
    ex = np.fromiter((p.exists() for p in ps), dtype=bool, count=len(ps))
    before = len(out)
    out = out[ex].copy()
    stats["drop_missing_image"] = before - len(out)

    out["article_type"] = out["articleType"].map(article_ds_to_key)
    out["is_other"] = out["article_type"].isna()
    out.loc[out["is_other"], "article_type"] = "other"

    out["base_colour"] = out["baseColour"].map(colour_ds_to_key)
    before = len(out)
    out = out[out["base_colour"].notna()].copy()
    stats["drop_unmapped_colour"] = before - len(out)

    out["usage"] = out["usage"].map(usage_ds_to_key)
    out["usage_valid"] = out["usage"].notna()
    stats["usage_masked"] = int((~out["usage_valid"]).sum())

    other = out[out["is_other"]]
    stats["other_before_cap"] = int(len(other))
    if len(other) > other_max:
        keep = other.sample(n=other_max, random_state=seed)
        out = pd.concat([out[~out["is_other"]], keep]).sort_index()
    stats["other_after_cap"] = int(out["is_other"].sum())

    out = out.reset_index(drop=True)
    if max_samples is not None and len(out) > max_samples:
        out = out.sample(n=max_samples, random_state=seed).sort_index().reset_index(drop=True)

    out["image_path"] = [images_dir / f"{i}.jpg" for i in out["id"]]
    stats["rows_kept"] = int(len(out))
    return out, stats


frame, clean_stats = clean_fashion(styles, images_dir, max_samples=MAX_SAMPLES)
display(pd.Series(clean_stats, name="nilai").to_frame())
print("\nfinal rows:", len(frame))

raw rows: 44424


,nilai
drop_bad_id,0
drop_missing_core,15
drop_gender,1485
drop_missing_image,5
drop_unmapped_colour,229
usage_masked,3488
other_before_cap,13069
other_after_cap,4000
rows_kept,33621



final rows: 33621


In [5]:
print("article_type:\n", frame["article_type"].value_counts().to_string())
print("\nbase_colour:\n", frame["base_colour"].value_counts().to_string())
print("\nusage (masked = ''):\n", frame["usage"].fillna("<masked>").value_counts().to_string())
print("\nusage_valid:", int(frame["usage_valid"].sum()), "/", len(frame))

article_type:
 article_type
tshirt          6428
other           4000
shirt           3136
casual_shoes    2807
watch           2499
sports_shoes    2030
handbag         1753
top             1533
heels           1276
sunglasses      1006
flip_flops       891
sandals          847
belt             811
backpack         722
formal_shoes     637
jeans            558
trousers         523
flats            489
shorts           456
track_pants      304
sweatshirt       282
sweater          277
jacket           252
skirt            104

base_colour:
 base_colour
black        8058
white        4529
blue         3945
grey         3447
brown        2902
red          2168
green        1865
navy_blue    1411
purple       1389
pink         1256
beige        1075
yellow        992
orange        371
multi         213

usage (masked = ''):
 usage
casual          26530
sports           3740
formal           2065
<masked>         1208
smart_casual       63
party              15

usage_valid: 32413 / 33621


## 3. Stratified split 80/10/10 (deterministik, seed 42)

In [6]:
def stratified_split(df, *, val_frac=0.1, test_frac=0.1, seed=SEED, stratify_col="article_type"):
    rng = np.random.default_rng(seed)
    tr, va, te = [], [], []
    for _, g in df.groupby(stratify_col, sort=True):
        idx = g.index.to_numpy().copy()
        rng.shuffle(idx)
        n = len(idx)
        if n < 3:
            tr.extend(idx.tolist())
            continue
        n_val = max(1, int(round(val_frac * n)))
        n_test = max(1, int(round(test_frac * n)))
        if n - n_val - n_test < 1:
            n_val, n_test = 0, 0
        va.extend(idx[:n_val].tolist())
        te.extend(idx[n_val:n_val + n_test].tolist())
        tr.extend(idx[n_val + n_test:].tolist())
    out = df.copy()
    out["split"] = "train"
    out.loc[va, "split"] = "val"
    out.loc[te, "split"] = "test"
    return out


frame = stratified_split(frame, seed=SEED)
print("split sizes:", frame["split"].value_counts().to_dict())

# verifikasi tidak ada kebocoran id antar split
sets = {k: set(v) for k, v in frame.groupby("split")["id"].apply(list).items()}
print("overlap train&val:", len(sets["train"] & sets["val"]),
      "| train&test:", len(sets["train"] & sets["test"]),
      "| val&test:", len(sets["val"] & sets["test"]))

ct = pd.crosstab(frame["article_type"], frame["split"]).reindex(columns=["train", "val", "test"], fill_value=0)
display((ct.div(ct.sum(axis=1), axis=0) * 100).round(1).head(12))

split sizes:

 {'train': 26895, 'test': 3363, 'val': 3363}
overlap train&val: 0 | train&test: 0 | val&test: 0


split,train,val,test
article_type,,,
backpack,80.1,10.0,10.0
belt,80.0,10.0,10.0
casual_shoes,80.0,10.0,10.0
flats,80.0,10.0,10.0
flip_flops,80.0,10.0,10.0
formal_shoes,79.9,10.0,10.0
handbag,80.0,10.0,10.0
heels,79.9,10.0,10.0
jacket,80.2,9.9,9.9


## 4. Cache gambar → `fashion_128.npy`

Semua gambar diproses **sekali** dengan `preprocess_pil` (square center-crop + resize 128,
sama persis dengan backend). Jika cache sudah ada dan cocok, pembuatan dilewati
(±13 menit di CPU). Set `FORCE_REBUILD=True` untuk membuat ulang.

In [7]:
import os
from tqdm import tqdm
from preprocess import preprocess_path

npy_path = PROCESSED_DIR / f"fashion_{IMG_SIZE}.npy"


def cache_images(df, npy_path, img_size):
    n = len(df)
    arr = np.lib.format.open_memmap(npy_path, mode="w+", dtype=np.uint8,
                                    shape=(n, img_size, img_size, 3))
    kept, failures, write = [], [], 0
    for pos, row in enumerate(tqdm(df.itertuples(index=False), total=n, desc="cache 128px")):
        try:
            img = preprocess_path(row.image_path, size=img_size)
        except Exception as exc:
            failures.append({"id": int(row.id), "error": str(exc)})
            continue
        arr[write] = img
        kept.append(pos)
        write += 1
    arr.flush()
    del arr
    if write < n:
        src = np.lib.format.open_memmap(npy_path, mode="r")
        tmp = npy_path.with_name(npy_path.stem + "_compact.npy")
        dst = np.lib.format.open_memmap(tmp, mode="w+", dtype=np.uint8,
                                        shape=(write, img_size, img_size, 3))
        for s in range(0, write, 1024):
            dst[s:s + 1024] = src[s:s + 1024]
        dst.flush()
        del dst, src
        os.replace(tmp, npy_path)
    return kept, failures


can_reuse = False
if npy_path.exists() and not FORCE_REBUILD and MAX_SAMPLES is None:
    try:
        prev = pd.read_csv(PROCESSED_DIR / "fashion_index.csv")
        can_reuse = len(prev) == len(frame) and prev["id"].tolist() == frame["id"].tolist()
    except Exception:
        can_reuse = False

if can_reuse:
    failures = []
    print("Cache ditemukan & cocok -> lewati pembuatan. (FORCE_REBUILD=True untuk rebuild)")
else:
    frame = frame.reset_index(drop=True)
    kept, failures = cache_images(frame, npy_path, IMG_SIZE)
    frame = frame.iloc[kept].reset_index(drop=True)
    print("unreadable images:", len(failures))

arr = np.load(npy_path, mmap_mode="r")
print("npy:", arr.shape, arr.dtype, "| %.2f GB" % (os.path.getsize(npy_path) / 1e9))

Cache ditemukan & cocok -> lewati pembuatan. (FORCE_REBUILD=True untuk rebuild)
npy: (33621, 128, 128, 3) uint8 | 1.65 GB


## 5. Simpan indeks & split

In [8]:
def rel_posix(p):
    try:
        return Path(p).resolve().relative_to(cu.data_dir()).as_posix()
    except ValueError:
        return Path(p).as_posix()


index_df = frame[["id", "gender", "article_type", "base_colour", "usage",
                  "usage_valid", "is_other", "split"]].copy()
index_df.insert(0, "npy_index", np.arange(len(index_df), dtype=np.int64))
index_df["usage"] = index_df["usage"].fillna("")
index_df["usage_valid"] = index_df["usage_valid"].astype(bool)
index_df["is_other"] = index_df["is_other"].astype(bool)
index_df["image_path"] = [rel_posix(p) for p in frame["image_path"]]
index_df = index_df[["npy_index", "id", "gender", "article_type", "base_colour",
                     "usage", "usage_valid", "is_other", "split", "image_path"]]

index_df.to_csv(PROCESSED_DIR / "fashion_index.csv", index=False)
for split in ("train", "val", "test"):
    index_df[index_df["split"] == split].to_csv(SPLITS_DIR / f"{split}.csv", index=False)

print("rows:", len(index_df))
print("split:", index_df["split"].value_counts().to_dict())
print("npy_index kontinu:", index_df["npy_index"].tolist() == list(range(len(index_df))))
display(index_df.head())
print("tersimpan:", cu.to_posix(PROCESSED_DIR / "fashion_index.csv"))

rows: 33621
split: {'train': 26895, 'test': 3363, 'val': 3363}
npy_index kontinu: True


,npy_index,id,gender,article_type,base_colour,usage,usage_valid,is_other,split,image_path
0,0,15970,Men,shirt,navy_blue,casual,True,False,train,fashionProduct/images/15970.jpg
1,1,39386,Men,jeans,blue,casual,True,False,train,fashionProduct/images/39386.jpg
2,2,59263,Women,watch,grey,casual,True,False,train,fashionProduct/images/59263.jpg
3,3,21379,Men,track_pants,black,casual,True,False,train,fashionProduct/images/21379.jpg
4,4,53759,Men,tshirt,grey,casual,True,False,train,fashionProduct/images/53759.jpg


tersimpan: C:/Kuliah/Semester 5/neurfit/ml/data/processed/fashion_index.csv


In [9]:
report = {
    "labels_version": labels.get("version"),
    "seed": SEED,
    "img_size": IMG_SIZE,
    "other_class_max_samples": OTHER_MAX,
    "max_samples": MAX_SAMPLES,
    "reused_cache": bool(can_reuse),
    "clean_stats": {k: int(v) for k, v in clean_stats.items()},
    "rows_final": int(len(index_df)),
    "split_sizes": {str(k): int(v) for k, v in index_df["split"].value_counts().items()},
    "article_counts": {str(k): int(v) for k, v in index_df["article_type"].value_counts().items()},
    "colour_counts": {str(k): int(v) for k, v in index_df["base_colour"].value_counts().items()},
    "usage_counts": {str(k): int(v) for k, v in index_df["usage"].replace("", "<masked>").value_counts().items()},
    "npy": {"path": cu.to_posix(npy_path), "shape": list(arr.shape), "dtype": str(arr.dtype)},
    "unreadable_images": len(failures),
}
(ARTIFACT_DIR / "fashion_prepare_report.json").write_text(
    json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print("report:", cu.to_posix(ARTIFACT_DIR / "fashion_prepare_report.json"))

report: C:/Kuliah/Semester 5/neurfit/ml/artifacts/fashion_prepare_report.json


## 6. Serah-terima ke modeling

Output siap dipakai notebook modeling:
- `fashion_128.npy` → gambar uint8 `(N,128,128,3)` skala 0–255 (normalisasi ada di dalam model).
- `fashion_index.csv` → `npy_index` selalu sinkron dengan baris `.npy`.
- `splits/{train,val,test}.csv` → subset indeks + label + `usage_valid`.

Cara load singkat:

```python
import numpy as np, pandas as pd
images = np.load("data/processed/fashion_128.npy", mmap_mode="r")
train = pd.read_csv("data/processed/splits/train.csv")
x = images[train["npy_index"].to_numpy()]      # (n, 128, 128, 3) uint8
```

> `ml/data/` dan `ml/artifacts/` **tidak** di-commit (lihat `.gitignore`); dataset diunduh terpisah.